# Assignment 1 — Training pipeline
**Louis de Perier — BBADBA 5B**

Trains one ridge-regularised linear regression **three ways** on the Ames Housing
data and writes everything the Gradio dashboard needs into `artifacts/`.

This is the only stage that trains anything.

In [ ]:
# Bootstrap: makes this notebook run in Google Colab AND under main.py.
# Pure Python - no ! or % magics, because `jupyter nbconvert --to script`
# turns those into get_ipython() calls that crash outside a kernel.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not Path("features.py").exists():
    REPO = "https://github.com/YOUR-USERNAME/daiist-assignment-1.git"  # <-- EDIT
    subprocess.run(["git", "clone", "-q", REPO, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch", "gradio", "plotly", "scikit-learn"], check=True)
    print("Colab ready, working in", os.getcwd())
else:
    print("Local/CI run, working in", os.getcwd())

## 1. Configuration

`features.py` and `models.py` stay as importable modules rather than being
pasted in here. That is deliberate: the fitted preprocessor is pickled, and
pickle stores the **class's module path**. If `Preprocessor` were defined in
this notebook it would be saved as `__main__.Preprocessor`, and the dashboard
would fail to unpickle it.

The two hyperparameters were chosen on the 2009 validation set. `alpha=100` was
the best ridge strength; `lr=0.05` is the largest stable learning rate, since
0.1 and above diverge to `nan`.

In [ ]:
import json, pickle, time
from pathlib import Path
import numpy as np
from sklearn.linear_model import Ridge

import features as F
import models as M

ROOT = Path.cwd()
ART = ROOT / "artifacts"; ART.mkdir(exist_ok=True)

ALPHA, LR, EPOCHS, SEED = 100.0, 0.05, 3000, 42
NAMES = ["baseline", "sklearn", "manual_torch", "standard_torch"]

## 2. Data, features, chronological split

`F.build()` runs the whole preparation: scope to arm's-length sales, fill
structural vs genuine missingness separately, map 17 ordinal quality scales to
ordered integers, derive age/interaction/cyclical features, split on `Yr Sold`
(2006–08 train, 2009 validation, 2010 test), then fit the preprocessor on
**training rows only** and apply it frozen to the other two splits.

In [ ]:
data = F.build()
X_tr, y_tr = data["X_train"], data["y_train"]
X_va, y_va = data["X_val"],   data["y_val"]
X_te, y_te = data["X_test"],  data["y_test"]

n = X_tr.shape[0]
# sklearn penalises a SUMMED loss, torch a MEAN loss  ->  wd = 2*alpha/n
weight_decay = 2 * ALPHA / n
print(f"train {X_tr.shape} | val {X_va.shape} | test {X_te.shape}")
print(f"weight_decay = 2*{ALPHA:.0f}/{n} = {weight_decay:.5f}")

params, timings, history = {}, {}, {}
params["baseline"] = np.concatenate([[y_tr.mean()], np.zeros(X_tr.shape[1])])

## 3. Implementation one — scikit-learn

`Ridge` does **not** iterate. It solves the problem in closed form, computing the
exact minimum directly. There is no learning rate and no epoch count because
there is no search. This is the reference the other two are measured against.

In [ ]:
t0 = time.perf_counter()
sk = Ridge(alpha=ALPHA).fit(X_tr, y_tr)
timings["sklearn"] = time.perf_counter() - t0
params["sklearn"] = np.concatenate([[sk.intercept_], sk.coef_])
print(f"fitted in {timings['sklearn']:.4f}s, intercept {sk.intercept_:.4f}")

## 4. Implementation two — manual PyTorch loop

Four steps per epoch, nothing hidden:

1. **forward** — `X @ w + b`, then mean squared error plus the L2 penalty
2. **backward** — `loss.backward()` walks the recorded graph and fills `w.grad`
3. **update** — `w -= lr * w.grad`, inside `no_grad()` so the update itself isn't recorded
4. **zero** — PyTorch *accumulates* gradients, so skipping `.zero_()` would make
   epoch 2 use epoch 1 + 2's gradients and the model would diverge

The bias is deliberately left out of the penalty, to match what sklearn does.

In [ ]:
t0 = time.perf_counter()
params["manual_torch"], history["manual_torch"] = M.fit_manual_l2(
    X_tr, y_tr, weight_decay, LR, EPOCHS, SEED)
timings["manual_torch"] = time.perf_counter() - t0
print(f"fitted in {timings['manual_torch']:.3f}s")
print("final training MSE:", round(history["manual_torch"][-1][1], 6))

## 5. Implementation three — standard `nn.Module` + `torch.optim`

Identical mathematics. `nn.Linear` owns the parameters, `MSELoss` is the same
formula in a class, `optimizer.step()` performs the same subtraction and
`optimizer.zero_grad()` clears the same gradients.

**The bug worth remembering:** `torch.optim.SGD` applies `weight_decay` to *every*
parameter including the bias, while sklearn's `Ridge` never penalises the
intercept. Log prices centre near 12.0, so decaying the bias toward zero gave an
R² of **−3.3** while leaving the slopes perfectly correct. Parameter groups fix it.

In [ ]:
t0 = time.perf_counter()
params["standard_torch"], history["standard_torch"] = M.fit_standard_l2(
    X_tr, y_tr, weight_decay, LR, EPOCHS, SEED)
timings["standard_torch"] = time.perf_counter() - t0
print(f"fitted in {timings['standard_torch']:.3f}s")
print("final training MSE:", round(history["standard_torch"][-1][1], 6))

## 6. Evaluation against a naive baseline

Metrics are reported in log space (what we optimised) and in dollars (what the
lender sees). `within_10pct` is the share of houses priced within 10% — under the
business framing, that is the population that could skip a $500 appraisal.

In [ ]:
results = {name: {split: M.metrics(y, M.predict(prm, X))
                  for split, X, y in [("train", X_tr, y_tr), ("val", X_va, y_va),
                                      ("test", X_te, y_te)]}
           for name, prm in params.items()}

print(f"{'model':<17}{'RMSE_log':>10}{'R2':>8}{'MedAPE':>9}{'<=10%':>8}{'RMSE $':>12}{'fit s':>8}")
for k in NAMES:
    m = results[k]["test"]
    print(f"{k:<17}{m['rmse_log']:>10.4f}{m['r2']:>8.3f}{m['med_ape']:>9.2f}"
          f"{m['within_10pct']:>8.1f}{m['rmse_dollars']:>12,.0f}{timings.get(k,0):>8.2f}")

## 7. Do the three agree?

Yes — to five decimal places. All three solve the same strongly convex problem,
which has exactly one minimum: sklearn computes it directly, the two PyTorch
versions walk to it. Agreement is evidence that all three are correct.

In [ ]:
ref = params["sklearn"]
for k in ["manual_torch", "standard_torch"]:
    d = np.abs(params[k] - ref)
    print(f"{k:<16} max |diff| {d.max():.6f}   corr {np.corrcoef(params[k], ref)[0,1]:.6f}")

## 8. Save artifacts

Each model reduces to 168 numbers. The dashboard loads these and never retrains.

In [ ]:
np.savez_compressed(ART / "params.npz", **params)
with open(ART / "preprocessor.pkl", "wb") as f:
    pickle.dump(data["pre"], f)

preds = {n: {s: M.predict(p, data[f"X_{s}"]).tolist() for s in ["train","val","test"]}
         for n, p in params.items()}
with open(ART / "results.json", "w") as f:
    json.dump({"metrics": results, "timings": timings, "history": history,
               "predictions": preds,
               "truth": {s: data[f"y_{s}"].tolist() for s in ["train","val","test"]},
               "feature_names": data["pre"].feature_names_,
               "config": {"alpha": ALPHA, "weight_decay": weight_decay,
                          "lr": LR, "epochs": EPOCHS, "seed": SEED}}, f)
for split in ["train", "test"]:
    data[f"raw_{split}"].to_csv(ART / f"raw_{split}.csv", index=False)

print("artifacts written:", sorted(p.name for p in ART.iterdir()))